# Выполнение ЛР №5: Классификация и регрессия

## Подключение библиотек

In [ ]:
# Импорт необходимых библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Импорт модулей sklearn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

# Дополнительные импорты для обработки данных
import warnings
warnings.filterwarnings('ignore')

## Настройка библиотек

In [ ]:
# Настройка numpy для воспроизводимости результатов
np.random.seed(42)

## Задание 1: Классификация kNN на датасете flame

### Формулировка

Выполнить классификацию методом k ближайших соседей на датасете flame:
1. Загрузить и разобрать данные из файла flame.txt
1. Разделить данные flame на обучающий и тестовый наборы
1. Оценить точность для разных значений k от 2 до 20 с использованием кросс-валидации
1. Построить график зависимости точности от k

### Решение

#### 1.1 Загрузка и парсинг данных flame

In [ ]:
# Загрузка данных из файла flame.txt
flame_data_path = '../ЛР (4)/Вариант 4/flame.txt'

# Чтение данных с разделителем табуляция
flame_data = pd.read_csv(flame_data_path, sep='\t', header=None, names=['x1', 'x2', 'class'])

print("Данные flame успешно загружены!")
print(f"Размер датасета: {flame_data.shape}")
print("\nПервые 10 строк:")
display(flame_data.head(10))

# Разделение данных на признаки (X) и метки классов (y)
X_flame = flame_data[['x1', 'x2']].values
y_flame = flame_data['class'].values

print(f"\nРазмер матрицы признаков X: {X_flame.shape}")
print(f"Размер вектора меток y: {y_flame.shape}")
print(f"Уникальные классы: {np.unique(y_flame)}")

#### 1.2 Разделение данных на обучающий и тестовый наборы

In [ ]:
# Разделение данных flame на обучающий и тестовый наборы
# Используем фиксированный random_state для воспроизводимости результатов
X_train, X_test, y_train, y_test = train_test_split(
    X_flame, y_flame, 
    test_size=0.3,  # 30% данных для тестирования
    random_state=42,  # Фиксированное значение для воспроизводимости
    stratify=y_flame  # Сохранение пропорций классов
)

print("Данные успешно разделены на обучающий и тестовый наборы!")
print(f"Размер исходного датасета: {X_flame.shape[0]} образцов")
print(f"Размер обучающего набора: {X_train.shape[0]} образцов ({X_train.shape[0]/X_flame.shape[0]*100:.1f}%)")
print(f"Размер тестового набора: {X_test.shape[0]} образцов ({X_test.shape[0]/X_flame.shape[0]*100:.1f}%)")

print("\nРаспределение классов в обучающем наборе:")
unique_train, counts_train = np.unique(y_train, return_counts=True)
for class_label, count in zip(unique_train, counts_train):
    print(f"  Класс {class_label}: {count} образцов ({count/len(y_train)*100:.1f}%)")

print("\nРаспределение классов в тестовом наборе:")
unique_test, counts_test = np.unique(y_test, return_counts=True)
for class_label, count in zip(unique_test, counts_test):
    print(f"  Класс {class_label}: {count} образцов ({count/len(y_test)*100:.1f}%)")

# Проверка корректности разделения
total_samples = X_train.shape[0] + X_test.shape[0]
print(f"\nПроверка: {X_train.shape[0]} + {X_test.shape[0]} = {total_samples} (исходно: {X_flame.shape[0]})")
assert total_samples == X_flame.shape[0], "Ошибка: потеря данных при разделении!"

#### 1.3 Оценка точности kNN для разных значений k

In [ ]:

# Оценка точности kNN для разных значений k от 2 до 20
from pandas import DataFrame


k_range = range(2, 21)  # k от 2 до 20
k_scores = []

print("Оценка точности kNN для разных значений k:")

# Цикл оценки для каждого k
accuracy_for_arr_k = []
for k in k_range:
    # Создание модели kNN
    knn = KNeighborsClassifier(n_neighbors=k)
    
    # Кросс-валидация с 5 фолдами
    cv_scores = cross_val_score(knn, X_train, y_train, cv=5, scoring='accuracy')
    
    # Сохранение среднего значения точности
    mean_accuracy = cv_scores.mean()
    std_accuracy = cv_scores.std()
    k_scores.append(mean_accuracy)
    
    accuracy_for_arr_k.append({
        'k': k,
        'Точность (среднее)': mean_accuracy,
        'Стандартное отклонение': std_accuracy,
    })
display(DataFrame(accuracy_for_arr_k).style.hide(axis='index'))

print(f"\nВсего оценено k значений: {len(k_scores)}")
print(f"Лучшая точность: {max(k_scores):.4f} при k = {k_range[k_scores.index(max(k_scores))]}")
print(f"Худшая точность: {min(k_scores):.4f} при k = {k_range[k_scores.index(min(k_scores))]}")

#### 1.4 Построение графика точности vs k

In [ ]:
# Построение графика зависимости точности от k
plt.figure(figsize=(12, 8))

# Основной график
plt.plot(k_range, k_scores, 'bo-', linewidth=2, markersize=8, label='Точность кросс-валидации')

# Выделение максимального значения
best_k = k_range[k_scores.index(max(k_scores))]
best_score = max(k_scores)
plt.plot(best_k, best_score, 'ro', markersize=12, label=f'Лучший результат (k={best_k})')

# Настройка графика
plt.xlabel('Количество соседей (k)', fontsize=14)
plt.ylabel('Точность классификации', fontsize=14)
plt.title('Зависимость точности kNN от количества соседей k\n(датасет flame)', fontsize=16)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)

# Установка диапазона осей
plt.xlim(1.5, 20.5)
plt.ylim(min(k_scores) - 0.02, max(k_scores) + 0.02)

# Настройка тиков на оси x
plt.xticks(range(2, 21, 2))

plt.tight_layout()
plt.show()

# Вывод статистики
print(f"\nСтатистика по результатам:")
print(f"Оптимальное значение k: {best_k}")
print(f"Максимальная точность: {best_score:.4f}")
print(f"Средняя точность по всем k: {np.mean(k_scores):.4f}")
print(f"Стандартное отклонение: {np.std(k_scores):.4f}")

#### Выводы по заданию 1

1. **Загрузка данных**: Успешно загружен датасет flame с двумя признаками (x1, x2) и двумя классами (1, 2)
1. **Разделение данных**: Данные flame успешно разделены на обучающий (70%) и тестовый (30%) наборы с сохранением пропорций классов
1. **Оценка kNN**: Проведена оценка точности для k от 2 до 20 с использованием 5-фолдовой кросс-валидации
1. **Визуализация**: Построен график зависимости точности от k, который показывает оптимальное значение k
1. **Результат**: Определено оптимальное значение k для данного датасета

## Задание 2: Визуализация данных с разделением на классы

### Формулировка

Визуализировать обучающие и тестовые данные с метками классов:
1. Создать диаграмму рассеяния с цветовым кодированием по классам
2. Различить обучающие и тестовые данные визуально
3. Добавить легенду с метками классов

### Решение

#### 2.1 Создание диаграммы рассеяния с классами

In [ ]:
# Определение цветов для классов
colors = {1: 'red', 2: 'blue'}
class_names = {1: 'Класс 1', 2: 'Класс 2'}

# Дополнительная детальная визуализация
plt.figure(figsize=(12, 8))

# Создание графика с разделением
for class_label in np.unique(y_flame):
    # Обучающие данные для каждого класса
    mask_train = y_train == class_label
    plt.scatter(X_train[mask_train, 0], X_train[mask_train, 1], 
               c=colors[class_label], alpha=0.7, s=70, 
               label=f'{class_names[class_label]} - Обучение ({np.sum(mask_train)} точек)', 
               marker='o', edgecolors='darkgray', linewidth=0.8)
    
    # Тестовые данные для каждого класса
    mask_test = y_test == class_label
    plt.scatter(X_test[mask_test, 0], X_test[mask_test, 1], 
               c=colors[class_label], alpha=1.0, s=100, 
               label=f'{class_names[class_label]} - Тест ({np.sum(mask_test)} точек)', 
               marker='^', edgecolors='black', linewidth=1.2)

plt.xlabel('Признак x1', fontsize=14)
plt.ylabel('Признак x2', fontsize=14)
plt.title('Детальная визуализация данных flame с разделением на классы и наборы', fontsize=16)
plt.legend(fontsize=11, loc='upper right', framealpha=0.9)
plt.grid(True, alpha=0.3)


plt.tight_layout()
plt.show()

#### Выводы по заданию 2

1. **Визуализация классов**: Создана диаграмма рассеяния с цветовым кодированием по классам (красный - класс 1, синий - класс 2)
1. **Различение наборов**: Обучающие данные показаны кругами, тестовые - треугольниками для четкого визуального различения

## Задание 3: Анализ результатов классификации

### Формулировка

Проанализировать результаты классификации kNN:
1. Выбрать оптимальное значение k на основе результатов кросс-валидации
1. Обучить финальную модель kNN с оптимальным k
1. Построить матрицу ошибок (confusion matrix)

### Решение

#### 3.1 Обучение оптимальной модели kNN

In [ ]:
# Выбор оптимального значения k на основе результатов кросс-валидации
optimal_k = k_range[k_scores.index(max(k_scores))]
optimal_accuracy = max(k_scores)

print("=== ВЫБОР ОПТИМАЛЬНОГО ЗНАЧЕНИЯ K ===")
print(f"Оптимальное значение k: {optimal_k}")
print(f"Максимальная точность кросс-валидации: {optimal_accuracy:.4f}")

# Обучение финальной модели kNN с оптимальным k
print("\n=== ОБУЧЕНИЕ ФИНАЛЬНОЙ МОДЕЛИ ===")
final_knn_model = KNeighborsClassifier(n_neighbors=optimal_k)
final_knn_model.fit(X_train, y_train)

print(f"Модель kNN успешно обучена с k = {optimal_k}")
print(f"Размер обучающего набора: {X_train.shape[0]} образцов")
print(f"Количество признаков: {X_train.shape[1]}")
print(f"Количество классов: {len(np.unique(y_train))}")

# Проверка обученной модели
print("\n=== ПАРАМЕТРЫ ОБУЧЕННОЙ МОДЕЛИ ===")
print(f"Алгоритм: {final_knn_model.algorithm}")
print(f"Метрика расстояния: {final_knn_model.metric}")
print(f"Количество соседей: {final_knn_model.n_neighbors}")
print(f"Веса: {final_knn_model.weights}")

# Получение предсказаний на тестовом наборе
y_pred = final_knn_model.predict(X_test)
y_pred_proba = final_knn_model.predict_proba(X_test)

print("\n=== ПРЕДСКАЗАНИЯ НА ТЕСТОВОМ НАБОРЕ ===")
print(f"Количество тестовых образцов: {len(y_test)}")
print(f"Количество предсказаний: {len(y_pred)}")
print(f"Уникальные предсказанные классы: {np.unique(y_pred)}")
print(f"Уникальные истинные классы: {np.unique(y_test)}")

# Предварительная оценка точности
test_accuracy = accuracy_score(y_test, y_pred)
print(f"\nТочность на тестовом наборе: {test_accuracy:.4f}")
print(f"Разница с кросс-валидацией: {abs(test_accuracy - optimal_accuracy):.4f}")

#### 3.2 Вычисление метрик производительности

In [ ]:
# Построение матрицы ошибок (confusion matrix)
cm = confusion_matrix(y_test, y_pred)

# Визуализация матрицы ошибок
plt.figure(figsize=(10, 8))

# Создание heatmap для матрицы ошибок
unique_classes = np.unique(y_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[f'Предсказан\nКласс {i}' for i in unique_classes],
            yticklabels=[f'Истинный\nКласс {i}' for i in unique_classes],
            cbar_kws={'label': 'Количество образцов'})

plt.title(f'Матрица ошибок для kNN (k={optimal_k})', fontsize=16)
plt.xlabel('Предсказанный класс', fontsize=14)
plt.ylabel('Истинный класс', fontsize=14)

# Добавление процентов в ячейки
total_samples = np.sum(cm)
for i in range(len(unique_classes)):
    for j in range(len(unique_classes)):
        percentage = cm[i, j] / total_samples * 100
        plt.text(j + 0.5, i + 0.7, f'({percentage:.1f}%)', 
                ha='center', va='center', fontsize=10, color='red')

plt.tight_layout()
plt.show()

# Анализ матрицы ошибок
print("\n--- АНАЛИЗ МАТРИЦЫ ОШИБОК ---")
total_correct = np.trace(cm)  # Сумма диагональных элементов
total_samples = np.sum(cm)
total_errors = total_samples - total_correct

print(f"Всего образцов: {total_samples}")
print(f"Правильно классифицировано: {total_correct} ({total_correct/total_samples*100:.1f}%)")
print(f"Неправильно классифицировано: {total_errors} ({total_errors/total_samples*100:.1f}%)")

#### Выводы по заданию 3

1. **Оптимальная модель**: Выбрано оптимальное значение k на основе кросс-валидации и обучена финальная модель kNN
3. **Матрица ошибок**: Построена и проанализирована confusion matrix

## Задание 4: Анализ ошибок классификации

### Формулировка

Проанализировать и визуализировать ошибки классификации kNN:
1. Идентифицировать неправильно классифицированные точки
1. Выделить эти точки на графике
1. Показать границы принятия решений

### Решение

#### 4.1 Идентификация неправильно классифицированных точек

In [ ]:

# Идентификация неправильно классифицированных точек
print("=== ИДЕНТИФИКАЦИЯ ОШИБОК КЛАССИФИКАЦИИ ===")

# Найти индексы неправильно классифицированных образцов
misclassified_mask = y_test != y_pred
misclassified_indices = np.where(misclassified_mask)[0]
correctly_classified_indices = np.where(~misclassified_mask)[0]

print(f"Всего тестовых образцов: {len(y_test)}")
print(f"Правильно классифицировано: {len(correctly_classified_indices)} ({len(correctly_classified_indices)/len(y_test)*100:.1f}%)")
print(f"Неправильно классифицировано: {len(misclassified_indices)} ({len(misclassified_indices)/len(y_test)*100:.1f}%)")

# Сохранение данных об ошибках для дальнейшего использования
misclassified_points = X_test[misclassified_indices]
misclassified_true_labels = y_test[misclassified_indices]
misclassified_pred_labels = y_pred[misclassified_indices]

print("Индексы неправильно классифицированных точек:")
print(misclassified_indices)


correctly_classified_points = X_test[correctly_classified_indices]
correctly_classified_labels = y_test[correctly_classified_indices]

print(f"\nДанные об ошибках сохранены для визуализации:")
print(f"- Неправильно классифицированные точки: {misclassified_points.shape}")
print(f"- Правильно классифицированные точки: {correctly_classified_points.shape}")

#### 4.2 Создание визуализации ошибок

In [ ]:
# Создание визуализации ошибок классификации
print("=== ВИЗУАЛИЗАЦИЯ ОШИБОК КЛАССИФИКАЦИИ ===")

# Создание фигуры
fig, axes = plt.subplots(1, 1, figsize=(10, 8))

# Определение цветов для классов и типов предсказаний
colors = {1: 'red', 2: 'blue'}
class_names = {1: 'Класс 1', 2: 'Класс 2'}

# График 1: Границы принятия решений
ax1 = axes

# Создание сетки для визуализации границ решений
h = 0.02  # Шаг сетки
x_min, x_max = X_test[:, 0].min() - 1, X_test[:, 0].max() + 1
y_min, y_max = X_test[:, 1].min() - 1, X_test[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

# Предсказания для всех точек сетки
mesh_points = np.c_[xx.ravel(), yy.ravel()]
Z = final_knn_model.predict(mesh_points)
Z = Z.reshape(xx.shape)

# Отображение границ решений
ax1.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)
ax1.contour(xx, yy, Z, colors='black', linewidths=0.5, alpha=0.5)

# Отображение тестовых точек
for class_label in np.unique(y_test):
    mask = y_test == class_label
    ax1.scatter(X_test[mask, 0], X_test[mask, 1], 
               c=colors[class_label], alpha=0.8, s=60, 
               label=f'{class_names[class_label]}', 
               marker='o', edgecolors='black', linewidth=0.5)

# Выделение ошибок
if len(misclassified_indices) > 0:
    ax1.scatter(misclassified_points[:, 0], misclassified_points[:, 1], 
               c='yellow', alpha=1.0, s=150, 
               label=f'Ошибки ({len(misclassified_indices)})', 
               marker='X', edgecolors='black', linewidth=2)

ax1.set_xlabel('Признак x1')
ax1.set_ylabel('Признак x2')
ax1.set_title('Границы принятия решений kNN')
ax1.legend()
ax1.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

#### Выводы по заданию 4

1. На тестовом наборе данных ошибок не выявлено.
1. Создана визуализация границы принятия решений kNN с выделением ошибок

## Задание 5: Предобработка данных

### Формулировка

* Выполнить предобработку  датасета  из  корневого каталога по аналогии с третьей лабораторной работой

### Решение

#### 5.1 Загрузка и изучение датасета

In [ ]:
# Загрузка данных из файла train.csv
train_data_path = 'house-prices-advanced/train.csv'

# Чтение данных
house_data = pd.read_csv(train_data_path)

print("=" * 60)
print("ЗАГРУЗКА И ИЗУЧЕНИЕ ДАТАСЕТА HOUSE PRICES")
print("=" * 60)

print(f"\nРазмер датасета: {house_data.shape}")
print(f"Количество строк: {house_data.shape[0]}")
print(f"Количество признаков: {house_data.shape[1]}")

print("\n" + "=" * 60)
print("ИНФОРМАЦИЯ О ДАТАСЕТЕ")
print("=" * 60)
house_data.info()

print("\n" + "=" * 60)
print("ПЕРВЫЕ 5 СТРОК ДАТАСЕТА")
print("=" * 60)
display(house_data.head())

print("\n" + "=" * 60)
print("СТАТИСТИКА ПО ЧИСЛЕННЫМ ПРИЗНАКАМ")
print("=" * 60)
display(house_data.describe())

# Разделение на численные и категориальные признаки
numerical_cols = house_data.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = house_data.select_dtypes(include=['object']).columns.tolist()

# Удаляем Id и SalePrice из численных (Id - идентификатор, SalePrice - целевая переменная)
numerical_cols.remove('Id')
numerical_cols.remove('SalePrice')

print("\n" + "=" * 60)
print("АНАЛИЗ ТИПОВ ПРИЗНАКОВ")
print("=" * 60)
print(f"\nЧисленные признаки ({len(numerical_cols)}):")
print(numerical_cols)

print(f"\nКатегориальные признаки ({len(categorical_cols)}):")
print(categorical_cols)

# Сохраняем целевую переменную отдельно
y_house = house_data['SalePrice'].copy()
print(f"\nЦелевая переменная SalePrice:")
print(f"  Минимум: {y_house.min():,.0f}")
print(f"  Максимум: {y_house.max():,.0f}")
print(f"  Среднее: {y_house.mean():,.0f}")
print(f"  Медиана: {y_house.median():,.0f}")


#### 5.2 Анализ пропущенных значений

In [ ]:
# Создаем копию данных для обработки
house_data_processed = house_data

print("=" * 60)
print("АНАЛИЗ ПРОПУЩЕННЫХ ЗНАЧЕНИЙ")
print("=" * 60)

# Подсчет пропущенных значений
missing_values = house_data_processed.isnull().sum()
missing_values = missing_values[missing_values > 0].sort_values(ascending=False)

if len(missing_values) > 0:
    missing_percent = (missing_values / len(house_data_processed)) * 100
    missing_df = pd.DataFrame({
        'Количество пропусков': missing_values,
        'Процент пропусков': missing_percent.round(2)
    })
    
    print(f"\nНайдено {len(missing_values)} признаков с пропущенными значениями:\n")
    display(missing_df)
    
    # Визуализация пропущенных значений
    plt.figure(figsize=(12, 8))
    top_missing = missing_df.head(20)  # Топ-20 признаков с пропусками
    plt.barh(range(len(top_missing)), top_missing['Процент пропусков'].values)
    plt.yticks(range(len(top_missing)), top_missing.index)
    plt.xlabel('Процент пропущенных значений (%)', fontsize=12)
    plt.title('Топ-20 признаков с пропущенными значениями', fontsize=14)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    # Анализ типов пропусков
    print("\n" + "=" * 60)
    print("АНАЛИЗ ПРОПУСКОВ ПО ТИПАМ ПРИЗНАКОВ")
    print("=" * 60)
    
    missing_numerical = [col for col in missing_values.index if col in numerical_cols]
    missing_categorical = [col for col in missing_values.index if col in categorical_cols]
    
    print(f"\nЧисленные признаки с пропусками ({len(missing_numerical)}):")
    if missing_numerical:
        for col in missing_numerical:
            print(f"  {col}: {missing_values[col]} пропусков ({missing_percent[col]:.2f}%)")
    else:
        print("  Нет")
    
    print(f"\nКатегориальные признаки с пропусками ({len(missing_categorical)}):")
    if missing_categorical:
        for col in missing_categorical:
            print(f"  {col}: {missing_values[col]} пропусков ({missing_percent[col]:.2f}%)")
    else:
        print("  Нет")
    
else:
    print("\n✓ Пропущенных значений не найдено!")


#### 5.3 Обработка пропущенных значений

In [ ]:
print("=" * 60)
print("ОБРАБОТКА ПРОПУЩЕННЫХ ЗНАЧЕНИЙ")
print("=" * 60)

# Словарь для отслеживания примененных стратегий
filling_strategies = {}

# 1. Обработка численных признаков
print("\n1. ОБРАБОТКА ЧИСЛЕННЫХ ПРИЗНАКОВ")
print("-" * 60)

for col in numerical_cols:
    if col in house_data_processed.columns and house_data_processed[col].isnull().sum() > 0:
        missing_count = house_data_processed[col].isnull().sum()
        
        # Для численных признаков используем медиану (более устойчива к выбросам)
        median_value = house_data_processed[col].median()
        house_data_processed[col].fillna(median_value, inplace=True)
        
        filling_strategies[col] = f'Медиана: {median_value:.2f}'
        print(f"  {col}: заполнено {missing_count} пропусков медианой ({median_value:.2f})")

# 2. Обработка категориальных признаков
print("\n2. ОБРАБОТКА КАТЕГОРИАЛЬНЫХ ПРИЗНАКОВ")
print("-" * 60)

# Список признаков, где "NA" означает "None" (согласно описанию)
# Это признаки, связанные с отсутствием объектов (нет подвала, нет гаража и т.д.)
na_means_none = [
    'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'PoolQC', 'Fence', 'MiscFeature'
]

for col in categorical_cols:
    if col in house_data_processed.columns:
        # Проверяем, есть ли реальные пропуски (NaN)
        nan_count = house_data_processed[col].isnull().sum()
        
        # Проверяем, есть ли строковые "NA"
        na_string_count = 0
        if house_data_processed[col].dtype == 'object':
            na_string_count = (house_data_processed[col] == 'NA').sum()
        
        if nan_count > 0:
            # Если это признак, где "NA" означает "None", заменяем NaN на "None"
            if col in na_means_none:
                house_data_processed[col].fillna('None', inplace=True)
                filling_strategies[col] = "Заполнено 'None' (отсутствие объекта)"
                print(f"  {col}: заполнено {nan_count} пропусков значением 'None'")
            else:
                # Для остальных категориальных признаков используем моду
                mode_value = house_data_processed[col].mode()
                if len(mode_value) > 0:
                    house_data_processed[col].fillna(mode_value[0], inplace=True)
                    filling_strategies[col] = f"Мода: {mode_value[0]}"
                    print(f"  {col}: заполнено {nan_count} пропусков модой ('{mode_value[0]}')")
                else:
                    # Если нет моды, используем 'Unknown'
                    house_data_processed[col].fillna('Unknown', inplace=True)
                    filling_strategies[col] = "Заполнено 'Unknown'"
                    print(f"  {col}: заполнено {nan_count} пропусков значением 'Unknown'")
        
        # Заменяем строковые "NA" на "None" для признаков, где это уместно
        if na_string_count > 0 and col in na_means_none:
            house_data_processed[col] = house_data_processed[col].replace('NA', 'None')
            print(f"  {col}: заменено {na_string_count} значений 'NA' на 'None'")

# 3. Финальная проверка
print("\n" + "=" * 60)
print("ФИНАЛЬНАЯ ПРОВЕРКА ПРОПУЩЕННЫХ ЗНАЧЕНИЙ")
print("=" * 60)

final_missing = house_data_processed.isnull().sum()
final_missing = final_missing[final_missing > 0]

if len(final_missing) == 0:
    print("✓ Все пропущенные значения успешно обработаны!")
else:
    print(f"⚠️ Остались пропуски в следующих признаках ({len(final_missing)}):")
    for col, count in final_missing.items():
        print(f"  {col}: {count} пропусков")

print("\n" + "=" * 60)
print("ИНФОРМАЦИЯ О ДАТАСЕТЕ ПОСЛЕ ОБРАБОТКИ ПРОПУСКОВ")
print("=" * 60)
house_data_processed.info()


#### 5.4 Преобразование категориальных данных


In [ ]:
print("=" * 60)
print("ПРЕОБРАЗОВАНИЕ КАТЕГОРИАЛЬНЫХ ДАННЫХ")
print("=" * 60)

# Определяем порядковые признаки (качества и условия)
# Эти признаки имеют естественный порядок и могут быть закодированы численно
ordinal_features = {
    'ExterQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'ExterCond': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtQual': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtCond': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtExposure': ['None', 'No', 'Mn', 'Av', 'Gd'],
    'BsmtFinType1': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'BsmtFinType2': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'HeatingQC': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'KitchenQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'FireplaceQu': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageFinish': ['None', 'Unf', 'RFn', 'Fin'],
    'GarageQual': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageCond': ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'PoolQC': ['None', 'Fa', 'TA', 'Gd', 'Ex'],
    'Fence': ['None', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv'],
    'Functional': ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
    'LandSlope': ['Sev', 'Mod', 'Gtl'],
    'LotShape': ['IR3', 'IR2', 'IR1', 'Reg'],
    'Utilities': ['ELO', 'NoSeWa', 'NoSewr', 'AllPub'],
    'LandContour': ['Low', 'Bnk', 'HLS', 'Lvl']
}

print("\n1. ОРДИНАЛЬНОЕ КОДИРОВАНИЕ (для порядковых признаков)")
print("-" * 60)

# Применяем ординальное кодирование для порядковых признаков
ordinal_encoders = {}
for col, categories in ordinal_features.items():
    if col in house_data_processed.columns:
        # Создаем маппинг категорий в числа
        category_map = {cat: idx for idx, cat in enumerate(categories)}
        
        # Применяем кодирование
        house_data_processed[col] = house_data_processed[col].map(category_map)
        
        # Если есть значения, которых нет в списке, заполняем медианой
        if house_data_processed[col].isnull().sum() > 0:
            median_val = house_data_processed[col].median()
            house_data_processed[col].fillna(median_val, inplace=True)
        
        house_data_processed[col] = house_data_processed[col].astype(int)
        ordinal_encoders[col] = category_map
        
        print(f"  {col}: закодировано {len(categories)} категорий")

# Обновляем списки признаков
categorical_cols_remaining = [col for col in categorical_cols 
                             if col in house_data_processed.columns 
                             and col not in ordinal_features.keys()]

print(f"\nОсталось категориальных признаков для кодирования: {len(categorical_cols_remaining)}")

print("\n2. ONE-HOT КОДИРОВАНИЕ (для номинальных признаков)")
print("-" * 60)

# Применяем One-Hot Encoding для остальных категориальных признаков
if len(categorical_cols_remaining) > 0:
    # Используем get_dummies для One-Hot Encoding
    # drop_first=True для избежания мультиколлинеарности
    encoded_cols = pd.get_dummies(
        house_data_processed[categorical_cols_remaining], 
        prefix=categorical_cols_remaining,
        drop_first=True,
        dummy_na=False,
        dtype=int
    )
    
    # Удаляем исходные категориальные столбцы
    house_data_processed = house_data_processed.drop(columns=categorical_cols_remaining)
    
    # Добавляем закодированные столбцы
    house_data_processed = pd.concat([house_data_processed, encoded_cols], axis=1)
    
    print(f"  Создано {len(encoded_cols.columns)} новых бинарных признаков")
    print(f"  Примеры новых признаков: {list(encoded_cols.columns[:5])}...")
else:
    print("  Нет категориальных признаков для One-Hot кодирования")

# 3. Финальная проверка
print("\n" + "=" * 60)
print("ФИНАЛЬНАЯ ПРОВЕРКА ПРЕОБРАЗОВАНИЙ")
print("=" * 60)

print(f"\nРазмер датасета после преобразований: {house_data_processed.shape}")
print(f"  Строк: {house_data_processed.shape[0]}")
print(f"  Столбцов: {house_data_processed.shape[1]}")

# Проверяем, что все категориальные признаки преобразованы
remaining_categorical = house_data_processed.select_dtypes(include=['object']).columns.tolist()
if 'Id' in remaining_categorical:
    remaining_categorical.remove('Id')

if len(remaining_categorical) == 0:
    print("\n✓ Все категориальные признаки успешно преобразованы!")
else:
    print(f"\n⚠️ Остались категориальные признаки ({len(remaining_categorical)}):")
    print(remaining_categorical)

# Удаляем Id, если он есть (не нужен для модели)
if 'Id' in house_data_processed.columns:
    house_data_processed = house_data_processed.drop(columns=['Id'])


### Выводы

1. **Загрузка данных**: Успешно загружен датасет House Prices с 1460 строками и 81 признаком. Целевая переменная SalePrice сохранена отдельно.

2. **Анализ пропущенных значений**: 
   - Проанализированы все пропущенные значения по каждому признаку
   - Учтена особенность датасета: "NA" в некоторых полях означает категорию "None" (отсутствие объекта), а не пропуск
   - Визуализированы топ-20 признаков с наибольшим количеством пропусков

3. **Обработка пропущенных значений**:
   - **Численные признаки**: заполнены медианой (более устойчива к выбросам)
   - **Категориальные признаки**: 
     - Для признаков, где "NA" означает отсутствие объекта (Alley, BsmtQual, GarageType и др.) → заполнено значением "None"
     - Для остальных категориальных признаков → заполнено модой (наиболее частым значением)
   - Все пропущенные значения успешно обработаны

4. **Преобразование категориальных данных**:
   - **Ординальное кодирование**: применено для порядковых признаков (качества, условия) с естественным порядком (Ex > Gd > TA > Fa > Po)
   - **One-Hot Encoding**: применено для номинальных категориальных признаков (MSZoning, Neighborhood, HouseStyle и др.)
   - Все категориальные признаки успешно преобразованы в численные

5. **Результат**: Данные полностью подготовлены для дальнейшего анализа - отбора признаков и построения регрессионной модели (Эластичная сеть).


## Задание 6

### Формулировка

* Выполнить отбор переменных для регрессии

### Решение

## Задание 7

### Формулировка

* Создать регрессионную модель используя метод "Эластичная сеть"

### Решение

## Задание 8

### Формулировка

* Для модели из задания 7 вывести среднеквадратическую ошибку (RMSE)

### Решение